In [9]:
import getpass
import json
import requests

from base64 import b64encode
from pathlib import Path
from urllib.parse import urlparse
from bs4 import BeautifulSoup

data_url = "https://dap.ceda.ac.uk/badc/quest/data/qgsi/wp_d1_climate_scenarios/nc/"

output_dir = Path("../data/download")
output_dir.mkdir(parents=True, exist_ok=True)

filename = Path(urlparse(data_url).path).name
output_file = output_dir / filename


In [10]:
class HTTPDownloaderGWS():
    
    def __init__(self, url):
        self._session = requests.Session()
        self._soup = self._get_soup(url)

    def _get_soup(self, url) -> BeautifulSoup:
        with self._session.get(url) as response:
            return BeautifulSoup(response.text, "html.parser")
        
    def _is_directory(self) -> bool:

        if self._soup.title and self._soup.title.string:
            return self._soup.title.string.startswith("Index of")
        return False
    
    def _directory_contents(self)-> dict:
        
        contents = []
        for link in self._soup.find_all("a"):
            href = link.get("href")
            # skip any things that are not needed like order query
            if "Parent Directory" in link.text:
                continue
            if "?" in href:
                continue

            contents.append(href)
            print("-", href)
        
        return contents
        
gws = HTTPDownloaderGWS(data_url)
print(gws._directory_contents())

- ../
- moves.txt
- p1.py
- p2.py
['../', 'moves.txt', 'p1.py', 'p2.py']


In [ ]:
with requests.get(data_url, stream=True, timeout=30) as r:
    r.raise_for_status()

    with open(output_file, "wb") as f:
        for chunk in r.iter_content(chunk_size=8192):
            if chunk:
                f.write(chunk)
